# LangChain: Models, Messages & Structured Output

## Outline
* نصب و setup
* مقایسه LangChain با litellm / openai یا ollama مستقیم
* init_chat_model — اتصال به هر provider
* Messages — انواع پیام‌ها
* Prompt Templates
* Streaming
* Structured Output با Pydantic
* Token Usage


## ۱. نصب

In [1]:
# نصب پکیج‌های مورد نیاز
# pip install -U langchain langchain-openai langchain-ollama python-dotenv


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

## ۲. چرا LangChain؟

- **LangChain**: علاوه بر یکپارچگی provider، ابزارهای کامل برای ساخت agent، memory، RAG، و evaluation


In [4]:
# روش قدیمی — openai مستقیم
# from openai import OpenAI
# client = OpenAI()
# response = client.chat.completions.create(
#     model="gpt-4o",
#     messages=[{"role": "user", "content": "Hello"}]
# )
# print(response.choices[0].message.content)

# روش LangChain — یکسان برای همه providerها
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.2", model_provider="openai", temperature=0)
response = model.invoke("Hello! What can you do?")
print(response.content)


I can help with a wide range of tasks—here are the main things I’m good at:

- **Answer questions & explain concepts:** science, history, math, programming, writing, etc.
- **Writing & editing:** emails, essays, reports, resumes, cover letters, summaries, tone rewrites, grammar.
- **Brainstorming & planning:** ideas, outlines, study plans, project plans, trip itineraries, meal plans.
- **Programming help:** code examples, debugging, explaining errors, designing algorithms, SQL, APIs.
- **Data & analysis:** interpret tables, basic statistics, compare options, build checklists/decision frameworks.
- **Working with images:** describe what’s in an image, extract text, help interpret charts/diagrams, troubleshoot visual issues.
- **Role-play & practice:** interview prep, difficult conversations, presentations, language practice.
- **Recommendations:** books, tools, learning resources, workflows (with constraints you give).

If you tell me what you’re trying to accomplish and any constraints

## ۳. init_chat_model — اتصال به هر Provider

مهم‌ترین تغییر نسبت به LangChain قدیمی: دیگه نیازی به import جداگانه برای هر provider نیست.


In [6]:
from langchain.chat_models import init_chat_model

# Ollama
model_ollama = init_chat_model("gemma3:4b", model_provider="ollama", temperature=0)

# Anthropic (نیاز به: pip install langchain-anthropic)
# model_anthropic = init_chat_model("claude-sonnet-4-6", model_provider="anthropic")

# Ollama — مدل local (نیاز به: pip install langchain-ollama)
# model_ollama = init_chat_model("llama3.2", model_provider="ollama")

# OpenAI-compatible (مثل litellm proxy یا vLLM)
# model_custom = init_chat_model(
#     model="your-model",
#     model_provider="openai",
#     base_url="http://localhost:11434/v1",
#     api_key="dummy"
# )

print(type(model_ollama))


<class 'langchain_ollama.chat_models.ChatOllama'>


In [8]:
a = model_ollama.invoke("سلام حالت چه طوره")
a.content

'سلام! من خوبم، ممنون که پرسیدی. شما چطورید؟ چه کاری میتونم براتون انجام بدم؟\n'

In [12]:
# پارامترهای مهم
model = init_chat_model(
    "gemma4:e4b",
    model_provider="ollama",
    temperature=0.7,       # خلاقیت (0 تا 2)
    max_tokens=500,        # حداکثر توکن خروجی
    timeout=30,            # timeout به ثانیه
    max_retries=3,         # تعداد retry در صورت خطا
)

## ۴. Messages — انواع پیام‌ها

در LangChain همه چیز با پیام کار می‌کنه. این مستقیماً با OpenAI API مپ می‌شه.


In [14]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# سه نوع اصلی پیام
system = SystemMessage("You are a helpful Persian-speaking assistant.")
human = HumanMessage("سلام! اسم من علی است.")
ai = AIMessage("سلام علی! چطور می‌تونم کمکت کنم؟")

# ارسال conversation کامل
conversation = [system, human, ai, HumanMessage("اسم من چیه؟")]
response = model.invoke(conversation)
print(response.content)
print(f"\nنوع response: {type(response)}")

اسم شما علی است. 👋

نوع response: <class 'langchain_core.messages.ai.AIMessage'>


In [16]:
# می‌تونید از dict هم استفاده کنید (مثل OpenAI API مستقیم)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]
response = model.invoke(messages)
print(response.content)


4


In [18]:
response

AIMessage(content='4', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-06T10:32:20.69121Z', 'done': True, 'done_reason': 'stop', 'total_duration': 580991200, 'load_duration': 184965400, 'prompt_eval_count': 29, 'prompt_eval_duration': 359278200, 'eval_count': 2, 'eval_duration': 35466600, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e9c7d-a38b-7021-a5ab-f5e90230aee0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 2, 'total_tokens': 31})

In [20]:
# metadata و token usage در response
response = model.invoke("Tell me a joke")
print(f"Content: {response.content}")
print(f"Model: {response.response_metadata.get('model_name', 'N/A')}")
print(f"Usage: {response.usage_metadata}")


Content: Why don't scientists trust atoms?

Because they make up everything! 😄
Model: gemma4:e4b
Usage: {'input_tokens': 20, 'output_tokens': 17, 'total_tokens': 37}


## ۵. Prompt Templates

Prompt template مثل یه تابع است که ورودی می‌گیره و prompt آماده برمی‌گردونه.


In [22]:
from langchain_core.prompts import ChatPromptTemplate

# ساده‌ترین حالت
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that translates {input_language} to {output_language}.
     You must only translate the text with no description, no explanation, no additional text."""),
    ("human", "{text}")
])

# پر کردن template
messages = prompt.invoke({
    "input_language": "English",
    "output_language": "Persian",
    "text": "Hello, how are you?"
})
print(messages)


messages=[SystemMessage(content='You are a helpful assistant that translates English to Persian.\n     You must only translate the text with no description, no explanation, no additional text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]


In [24]:
response = model.invoke(messages)
print(response.content)

سلام، حال شما چطوره؟


In [26]:
messages = prompt.invoke({
    "input_language": "Persian",
    "output_language": "Arabic",
    "text": "اسم شما چیه و چند سالتونه؟"
})
response = model.invoke(messages)
print(response.content)

ما اسمي وما عمري؟


## 5 . LCEL: LangChain Expression Language

**Building a `chain` with the `|` operator in LCEL:** Connects the prompt and model, then invokes the chain with a dictionary of prompt variables and prints the model's final output.

In [30]:
# استفاده در یک chain با LCEL (pipe operator)
chain = prompt | model

response = chain.invoke({
    "input_language": "English",
    "output_language": "Persian",
    "text": "LangChain is a powerful framework for building LLM applications."
})
print(response.content)


لان‌چین یک فریم‌ورک قدرتمند برای ساخت برنامه‌های مبتنی بر مدل‌های زبان بزرگ (LLM) است.


In [36]:
# Few-shot prompt — دادن مثال به مدل
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analyzer. Reply with only: POSITIVE, NEGATIVE, or NEUTRAL"),
    ("human", "I love this product!"),
    ("ai", "POSITIVE"),
    ("human", "This is terrible."),
    ("ai", "NEGATIVE"),
    ("human", "It's okay, nothing special."),
    ("ai", "NEUTRAL"),
    ("human", "{text}"),
])

chain = few_shot_prompt | model
result = chain.invoke({"text": "Best purchase I've made this year!"})
print(result.content)


POSITIVE


In [38]:
result = chain.invoke("Best purchase I've made this year!")
print(result.content)

POSITIVE


## ۶. Streaming

با streaming می‌تونید توکن‌ها رو همزمان با تولید نشون بدید — مثل ChatGPT.


In [40]:
# streaming ساده
print("خروجی streaming:")
for chunk in model.stream("Write a haiku about programming"):
    print(chunk.content, end="", flush=True)
print()  # newline


خروجی streaming:
Code lines brightly gleam,
Logic flows, a sweet rhythm,
Program starts to run.


In [42]:
# batch — ارسال چند request موازی
responses = model.batch([
    "What is Python?",
    "What is JavaScript?",
    "What is Rust?",
])
for i, r in enumerate(responses):
    print(f"Response {i+1}: {r.content[:80]}...")

Response 1: This is a fantastic question, because Python is one of the most popular and usef...
Response 2: This is one of the most popular and important questions in technology. Because J...
Response 3: Rust is a modern, high-performance, multi-paradigm programming language designed...


## 7. Structured Output with Pydantic

Instead of parsing text, force the model to directly output structured data.
This replaces the legacy `ResponseSchema` and `StructuredOutputParser`.

In [43]:
from pydantic import BaseModel, Field
from typing import List

# تعریف schema با Pydantic
class ProductReview(BaseModel):
    """اطلاعات استخراج‌شده از یک نظر محصول"""
    gift: bool = Field(description="آیا محصول به عنوان هدیه خریداری شده؟")
    delivery_days: int = Field(description="چند روز تحویل داد؟ اگر نبود -1")
    price_value: List[str] = Field(description="جملاتی درباره قیمت یا ارزش محصول")

# ساخت model با structured output
structured_model = model.with_structured_output(ProductReview)

In [46]:
review = """
This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze, 
windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. 
I think my wife liked it so much she was speechless. I paid 35 dollars for this for the tornado 
setting alone but you can get it for about half the price!
"""

result = structured_model.invoke(f"Extract info from this review: {review}")
print(f"Gift: {result.gift}")
print(f"Delivery days: {result.delivery_days}")
print(f"Price comments: {result.price_value}")
print(f"\nType: {type(result)}")


Gift: True
Delivery days: 2
Price comments: ['$35 (for tornado setting)', 'about half the price (current deal)']

Type: <class '__main__.ProductReview'>


In [76]:
# مثال دیگر: استخراج اطلاعات ساختاریافته
class MovieInfo(BaseModel):
    """اطلاعات فیلم"""
    title: str = Field(description="عنوان فیلم")
    year: int = Field(description="سال ساخت")
    director: str = Field(description="کارگردان")
    rating: float = Field(description="امتیاز از 10")

movie_model = model.with_structured_output(MovieInfo)
result = movie_model.invoke("Tell me about the movie Inception")
print(result)
print(f"\nTitle: {result.title}, Year: {result.year}")


title='Inception' year=2010 director='Christopher Nolan' rating=4.5

Title: Inception, Year: 2010


## ۸. Token Usage — ردیابی مصرف

برای کنترل هزینه مهمه.


In [90]:
from langchain_core.callbacks import get_usage_metadata_callback

model_gpt = init_chat_model("gpt-4o", model_provider="openai")

with get_usage_metadata_callback() as cb:
    model_gpt.invoke("Hello!")
    model_gpt.invoke("What is the capital of France?")

print(cb.usage_metadata)

{'gpt-4o-2024-08-06': {'output_tokens': 17, 'output_token_details': {'reasoning': 0}, 'input_token_details': {}, 'total_tokens': 40, 'input_tokens': 23}}


In [98]:
# usage در هر response هم موجوده
response = model.invoke("Explain machine learning in one sentence")
print(f"Input tokens:  {response.usage_metadata['input_tokens']}")
print(f"Output tokens: {response.usage_metadata['output_tokens']}")
print(f"Total tokens:  {response.usage_metadata['total_tokens']}")


Input tokens:  22
Output tokens: 319
Total tokens:  341
